# Lesson 03 - Agentic Design Patterns

In this lesson, we explore three foundational design patterns for building effective AI agents:

1. **Clear Agent Instructions** — Crafting precise, role-defining prompts that guide agent behavior
2. **Structured Output with Pydantic Models** — Ensuring agents return predictable, validated data
3. **Single Responsibility Agents** — Designing focused agents that each do one thing well

We'll apply each pattern to a **travel destination recommender** scenario, progressively building a system that can suggest destinations, check availability, and handle logistics.

## Setup

In [1]:
%pip install agent-framework azure-ai-projects azure-identity pydantic python-dotenv --quiet

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

c:\Users\lujan\AppData\Local\Programs\Python\Python312\Lib\site-packages\agent_framework\_skills.py:122: ExperimentalWarning: [SKILLS] SkillResource is experimental and may change or be removed in future versions without notice.
c:\Users\lujan\AppData\Local\Programs\Python\Python312\Lib\site-packages\agent_framework\_harness\_file_access.py:602: ExperimentalWarning: [HARNESS] AgentFileStore is experimental and may change or be removed in future versions without notice.


## Pattern 1: Clear Agent Instructions

The most impactful pattern is also the simplest: writing clear, detailed instructions for your agent.

Good instructions define:
- **Who** the agent is (persona and tone)
- **What** it should do (step-by-step responsibilities)
- **How** it should behave (constraints and style)

Below, we create a travel concierge agent with explicit instructions that shape every response it produces.

In [2]:
agent = provider.as_agent(
    name="TravelConcierge",
    instructions="""You are a luxury travel concierge named Alex. Your role is to:
1. Understand the traveler's preferences (budget, climate, activities)
2. Check destination availability before making recommendations
3. Provide detailed, personalized travel suggestions
4. Always mention visa requirements and best travel seasons
Be warm, professional, and enthusiastic about travel.""",
)

response = await agent.run(
    "I'd love a week-long vacation somewhere with great food and history. Budget around $2500."
)
print(response)

# How wonderful! I'd love to help you find the perfect culinary and historical escape. 

To give you the best recommendations, let me ask a few quick questions:

**1. Climate preference?**
   - Warm/tropical
   - Mild/temperate
   - Cool/mountainous
   - No preference

**2. When are you thinking of traveling?**
   - Specific months or flexible?

**3. Geography - any preferences or regions you'd like to explore?**
   - Europe, Asia, Latin America, Middle East, etc.?

**4. Activity style - what draws you to history/food?**
   - Museum/archaeological sites
   - Street food & local markets
   - Fine dining experiences
   - Cooking classes
   - Walking historic city neighborhoods

**5. Travel style?**
   - Budget-conscious (hostels, street eats)
   - Mid-range comfort (boutique hotels, mix of dining)
   - Luxury-leaning (within budget)

---

**Quick note:** Your $2500 budget is *excellent* for a week - that's roughly $357/day including flights, accommodation, and meals. This opens up some f

## Pattern 2: Structured Output with Pydantic Models

Free-form text is useful for conversation, but downstream systems need structured data.
By pairing **Pydantic models** with a **tool function**, we can:

- Define an exact schema for the agent's output
- Validate responses automatically
- Integrate agent results into application logic reliably

The key to enforcement is passing `response_format` when we run the agent. This forces the
model to return a validated `TravelRecommendations` object (available on `response.value`)
instead of free-form text. The `get_destination_details` tool also returns a typed
`DestinationRecommendation`, so the data stays structured end to end.


In [5]:
class DestinationRecommendation(BaseModel):
    destination: str
    available: bool
    best_season: str
    highlights: list[str]
    estimated_budget_usd: int


class TravelRecommendations(BaseModel):
    recommendations: list[DestinationRecommendation]
    personalized_note: str


@tool(approval_mode="never_require")
def get_destination_details(
    destination: Annotated[str, "The destination to look up"]
) -> DestinationRecommendation:
    """Get structured details about a vacation destination."""
    details = {
        "Barcelona": DestinationRecommendation(
            destination="Barcelona",
            available=True,
            best_season="May-Jun",
            highlights=["Beach", "Architecture", "Nightlife"],
            estimated_budget_usd=2000,
        ),
        "Tokyo": DestinationRecommendation(
            destination="Tokyo",
            available=True,
            best_season="Mar-Apr",
            highlights=["Culture", "Food", "Technology"],
            estimated_budget_usd=2500,
        ),
        "Cape Town": DestinationRecommendation(
            destination="Cape Town",
            available=False,
            best_season="Nov-Mar",
            highlights=["Nature", "Wine", "Adventure"],
            estimated_budget_usd=1800,
        ),
    }
    return details.get(
        destination,
        DestinationRecommendation(
            destination=destination,
            available=False,
            best_season="Unknown",
            highlights=[],
            estimated_budget_usd=0,
        ),
    )


# Structured outputs (`response_format`) requiere un modelo OpenAI que soporte
# json_schema forzado por el servidor; los deployments claude-* lo ignoran.
structured_provider = FoundryChatClient(
    project_endpoint=endpoint,
    model="gpt-5-mini",
    credential=DefaultAzureCredential(),
)

structured_agent = structured_provider.as_agent(
    name="StructuredTravelExpert",
    instructions="You are a travel expert. Recommend destinations based on traveler preferences. Use the get_destination_details tool.",
    tools=[get_destination_details],
)

# Passing `response_format` forces the agent to return a validated
# TravelRecommendations object instead of free-form text.
response = await structured_agent.run(
    "Recommend 3 destinations for a culture-loving traveler with a $2500 budget",
    options={"response_format": TravelRecommendations},
)

if response and response.value:
    result: TravelRecommendations = response.value
    for rec in result.recommendations:
        status = "Available" if rec.available else "Not available"
        print(f"{rec.destination} ({status})")
        print(f"  Best season: {rec.best_season}")
        print(f"  Highlights: {', '.join(rec.highlights)}")
        print(f"  Estimated budget: ${rec.estimated_budget_usd}")
        print()
    print(f"Note: {result.personalized_note}")
else:
    print("No validated structured response was returned.")
    print(response)


Mexico City, Mexico (Available)
  Best season: March–May or September–November (mild weather, fewer tourists)
  Highlights: Historic Center and Zócalo (pre-Hispanic & colonial layers), National Museum of Anthropology (world-class collection), Frida Kahlo's Casa Azul and Coyoacán neighborhood, Teotihuacan pyramids (day trip), Vibrant street food, markets (Mercado de La Merced, Roma/Condesa food scene)
  Estimated budget: $1400

Lisbon, Portugal (Available)
  Best season: April–June or September–October (pleasant weather, fewer crowds)
  Highlights: Belém (Jerónimos Monastery, Belém Tower, pastéis de nata), Alfama district and live Fado performances, Tram 28 ride and viewpoints (miradouros), Day trips to Sintra (palaces) and Cascais, Museums: MAAT, Calouste Gulbenkian, and rich maritime history
  Estimated budget: $2200

Istanbul, Turkey (Available)
  Best season: April–June or September–October (comfortable temperatures, blooming/harvest seasons)
  Highlights: Hagia Sophia, Blue Mosque,

## Pattern 3: Single Responsibility Agents

Complex tasks benefit from splitting work across multiple focused agents, each with a single responsibility:

- A **Destination Expert** that knows about places and availability
- A **Logistics Planner** that handles flights, hotels, and itineraries

This mirrors the software engineering principle of *separation of concerns* — each agent is easier to test, maintain, and improve independently.

In [4]:
destination_agent = provider.as_agent(
    name="DestinationExpert",
    tools=[get_destination_details],
    instructions="""You are a destination research specialist. Your only job is to:
1. Evaluate destinations based on traveler preferences
2. Check availability using the provided tool
3. Return a short ranked list with pros/cons
Do NOT discuss flights, hotels, or logistics — another agent handles that.""",
)

logistics_agent = provider.as_agent(
    name="LogisticsPlanner",
    instructions="""You are a travel logistics planner. Your only job is to:
1. Create a day-by-day itinerary for the chosen destination
2. Suggest flight and hotel options within the stated budget
3. Note visa requirements and travel insurance recommendations
Do NOT recommend destinations — another agent handles that.""",
)

# Step 1: Destination Expert picks the best options
dest_response = await destination_agent.run(
    "I want a week of culture and food for under $2500. Where should I go?"
)
print("=== Destination Expert ===")
print(dest_response)

# Step 2: Logistics Planner builds the trip plan
logistics_response = await logistics_agent.run(
    f"Plan a week-long trip based on this recommendation:\n{dest_response}"
)
print("\n=== Logistics Planner ===")
print(logistics_response)

=== Destination Expert ===
I'd love to help you find the perfect cultural and culinary destination! To give you the best recommendations, I need a bit more information:

1. **When are you planning to travel?** (month/season)
2. **Do you have any geographic preferences?** (e.g., Europe, Asia, Latin America, or open to anywhere?)
3. **Any specific cuisines or cultural interests?** (e.g., Italian food & history, Thai street food & temples, Mexican archaeology, etc.)
4. **Approximate home location or departure point?** (This helps me understand your budget context)

Once I understand your preferences better, I can look up specific destinations and give you a ranked list with pros and cons for each!

=== Logistics Planner ===
I appreciate you sharing that, but I need to clarify my role: **I'm the logistics planner, not the destination recommender.**

It looks like you've received a message from the **destination recommendation agent** — that's the right place to start! They'll help you narr

## Summary

In this lesson we applied three agentic design patterns to a travel recommender scenario:

| Pattern | Key Idea | Benefit |
|---|---|---|
| **Clear Instructions** | Define persona, responsibilities, and constraints up front | Consistent, on-brand agent behavior |
| **Structured Output** | Use Pydantic models as the response format | Validated, machine-readable results |
| **Single Responsibility** | Give each agent one focused job | Easier to test, maintain, and compose |

These patterns compose naturally — you can combine clear instructions with structured output inside a single-responsibility agent to build robust, production-ready systems.